# 5. Random Forest with TF-IDF
**Dataset:** CLINC150 — 151-class intent classification

**Approach:** TF-IDF (word + char n-grams) → **TruncatedSVD** (dense reduced features) → **RandomForestClassifier**

**Why SVD:** TF-IDF matrices here have tens of thousands of sparse dimensions. Training a random forest on that full sparse matrix is slow and memory-heavy; reducing to a few hundred dense components (LSA-style) keeps training feasible while preserving most signal.

**Expected Accuracy:** Often comparable to or slightly below linear TF-IDF models (~75–85% test — tune `n_components` and forest depth).

### Load Dataset

In [1]:
import os
import pandas as pd
from IPython.display import display
from datasets import load_dataset

test_dataset  = load_dataset('parquet', data_files={'test': os.path.join('..', 'data', 'test-00000-of-00001.parquet')})
train_dataset = load_dataset('parquet', data_files={'train': os.path.join('..', 'data', 'train-00000-of-00001.parquet')})
val_dataset   = load_dataset('parquet', data_files={'validation': os.path.join('..', 'data', 'validation-00000-of-00001.parquet')})

train_df = train_dataset['train'].to_pandas()
val_df   = val_dataset['validation'].to_pandas()
test_df  = test_dataset['test'].to_pandas()

print(f'Train: {len(train_df)} | Val: {len(val_df)} | Test: {len(test_df)}')
display(train_df.head(3))
display(val_df.head(3))
display(test_df.head(3))

Train: 10625 | Val: 3100 | Test: 5500


,text,intent
0,what are the steps for setting up direct depos...,108
1,how is a direct deposit set up,108
2,how would i go about setting up a direct deposit,108


,text,intent
0,"in spanish, meet me tomorrow is said how",61
1,"in french, how do i say, see you later",61
2,how do you say hello in japanese,61


,text,intent
0,how would you say fly in italian,61
1,what's the spanish word for pasta,61
2,how would they say butter in zambia,61


### Preprocessing

In [15]:
import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from nltk.tokenize import word_tokenize

# nltk.download('punkt'); nltk.download('stopwords')
# nltk.download('averaged_perceptron_tagger_eng'); nltk.download('wordnet')

def preprocess(sentences):
    base_stop_words = set(stopwords.words('english'))

    words_to_keep = {'what', 'where', 'who', 'why', 'how', 'when', 'do'}

    custom_stop_words = base_stop_words - words_to_keep

    allowed_symbols = {'?', '!'}

    lemmatizer = WordNetLemmatizer()
    result = []
    for sent in sentences:
        tokens = word_tokenize(sent)
        pos_tags = nltk.pos_tag(tokens)
        cleaned = []
        for token, pos in pos_tags:
            t = token.lower()
            if (t not in custom_stop_words) and (t.isalpha() or t in allowed_symbols):
                if t.isalpha():
                    if pos.startswith('NN'):   lemma = lemmatizer.lemmatize(t, 'n')
                    elif pos.startswith('VB'): lemma = lemmatizer.lemmatize(t, 'v')
                    elif pos.startswith('JJ'): lemma = lemmatizer.lemmatize(t, 'a')
                    else:                      lemma = lemmatizer.lemmatize(t)
                else:
                    lemma = t
                cleaned.append(lemma)
        result.append(' '.join(cleaned))
    return result

train_texts = preprocess(train_df['text'].tolist())
val_texts   = preprocess(val_df['text'].tolist())
test_texts  = preprocess(test_df['text'].tolist())

y_train = train_df['intent'].tolist()
y_val   = val_df['intent'].tolist()
y_test  = test_df['intent'].tolist()

print('Sample:', train_texts[0])

Sample: what step set direct deposit paycheck


### Vectorization — TF-IDF (word + char n-grams)

In [16]:
from sklearn.feature_extraction.text import TfidfVectorizer
import scipy.sparse as sp

word_vec = TfidfVectorizer(ngram_range=(1, 2), min_df=1, sublinear_tf=True, analyzer='word')
char_vec = TfidfVectorizer(ngram_range=(3, 5), min_df=1, sublinear_tf=True, analyzer='char_wb')

X_train_sparse = sp.hstack([
    word_vec.fit_transform(train_texts),
    char_vec.fit_transform(train_texts)
])
X_val_sparse  = sp.hstack([word_vec.transform(val_texts),  char_vec.transform(val_texts)])
X_test_sparse = sp.hstack([word_vec.transform(test_texts), char_vec.transform(test_texts)])

print(f'Sparse TF-IDF shape: {X_train_sparse.shape}')

Sparse TF-IDF shape: (10625, 43367)


### Dense features via TruncatedSVD (LSA)
Fit **only on training** rows, then transform validation and test. Adjust `n_components` (e.g. 200–600) if you want faster training vs more signal.

In [17]:
import numpy as np
from sklearn.decomposition import TruncatedSVD

N_COMPONENTS = 600
svd = TruncatedSVD(n_components=N_COMPONENTS, random_state=42)

X_train = svd.fit_transform(X_train_sparse).astype(np.float32)
X_val   = svd.transform(X_val_sparse).astype(np.float32)
X_test  = svd.transform(X_test_sparse).astype(np.float32)

var_exp = svd.explained_variance_ratio_.sum()
print(f'Dense shape after SVD: {X_train.shape}')
print(f'Explained variance ratio (sum): {var_exp:.4f}')

Dense shape after SVD: (10625, 600)
Explained variance ratio (sum): 0.6027


### Model Training — Random Forest
`class_weight='balanced'` helps with unequal samples per intent.

**Memory:** `GridSearchCV(n_jobs=-1)` **and** `RandomForestClassifier(n_jobs=-1)` together spawn *nested* parallel jobs and often exhaust RAM (Windows). This notebook uses **`n_jobs=1`** for both. **`max_leaf_nodes`** caps tree size for 151-way classification; **`max_depth=None`** was removed from the grid because unlimited depth blows memory.

In [20]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GridSearchCV

print('Running GridSearch over RF hyperparameters...')
param_grid = {
    'n_estimators': [150, 200],
    'max_depth': [35, 45],
    'min_samples_split': [5, 10],
    'min_samples_leaf': [2, 4]
}

gs = GridSearchCV(
    RandomForestClassifier(
        class_weight='balanced',
        random_state=42,
        n_jobs=1, # Keep this at 1 to prevent RAM spikes per tree
        max_features='sqrt',
        max_leaf_nodes=2048,
    ),
    param_grid, 
    cv=3, 
    scoring='accuracy', 
    n_jobs=8,  # Utilizes your 12th Gen i7 cores for parallel training
    verbose=3  # The native progress tracker
)

gs.fit(X_train, y_train)
print(f'Best params: {gs.best_params_}  |  CV Accuracy: {gs.best_score_:.4f}')
rf_model = gs.best_estimator_

Running GridSearch over RF hyperparameters...
Fitting 3 folds for each of 16 candidates, totalling 48 fits
Best params: {'max_depth': 45, 'min_samples_leaf': 2, 'min_samples_split': 5, 'n_estimators': 200}  |  CV Accuracy: 0.8020


### Evaluation

In [21]:
from sklearn.metrics import accuracy_score, classification_report

y_val_pred  = rf_model.predict(X_val)
y_test_pred = rf_model.predict(X_test)

print(f'Validation Accuracy : {accuracy_score(y_val, y_val_pred)*100:.2f}%')
print(f'Test Accuracy       : {accuracy_score(y_test, y_test_pred)*100:.2f}%')
print()
print('--- Validation Classification Report ---')
print(classification_report(y_val, y_val_pred))

Validation Accuracy : 77.94%
Test Accuracy       : 68.36%

--- Validation Classification Report ---
              precision    recall  f1-score   support

           0       0.46      0.60      0.52        20
           1       0.87      0.65      0.74        20
           2       0.88      0.75      0.81        20
           3       0.86      0.95      0.90        20
           4       0.76      0.80      0.78        20
           5       0.79      0.55      0.65        20
           6       0.94      0.75      0.83        20
           7       0.89      0.80      0.84        20
           8       0.83      0.95      0.88        20
           9       0.70      0.95      0.81        20
          10       0.83      1.00      0.91        20
          11       0.43      0.65      0.52        20
          12       0.86      0.90      0.88        20
          13       0.65      0.65      0.65        20
          14       1.00      1.00      1.00        20
          15       0.94      0.85  

### Model Testing

In [22]:
intent_mapping = {
    0: 'restaurant_reviews', 1: 'nutrition_info', 2: 'account_blocked', 3: 'oil_change_how', 4: 'time',
    5: 'weather', 6: 'redeem_rewards', 7: 'interest_rate', 8: 'gas_type', 9: 'accept_reservations',
    10: 'smart_home', 11: 'user_name', 12: 'report_lost_card', 13: 'repeat', 14: 'whisper_mode',
    15: 'what_are_your_hobbies', 16: 'order', 17: 'jump_start', 18: 'schedule_meeting', 19: 'meeting_schedule',
    20: 'freeze_account', 21: 'what_song', 22: 'meaning_of_life', 23: 'restaurant_reservation', 24: 'traffic',
    25: 'make_call', 26: 'text', 27: 'bill_balance', 28: 'improve_credit_score', 29: 'change_language',
    30: 'no', 31: 'measurement_conversion', 32: 'timer', 33: 'flip_coin', 34: 'do_you_have_pets',
    35: 'balance', 36: 'tell_joke', 37: 'last_maintenance', 38: 'exchange_rate', 39: 'uber',
    40: 'car_rental', 41: 'credit_limit', 42: 'oos', 43: 'shopping_list', 44: 'expiration_date',
    45: 'routing', 46: 'meal_suggestion', 47: 'tire_change', 48: 'todo_list', 49: 'card_declined',
    50: 'rewards_balance', 51: 'change_accent', 52: 'vaccines', 53: 'reminder_update', 54: 'food_last',
    55: 'change_ai_name', 56: 'bill_due', 57: 'who_do_you_work_for', 58: 'share_location', 59: 'international_visa',
    60: 'calendar', 61: 'translate', 62: 'carry_on', 63: 'book_flight', 64: 'insurance_change',
    65: 'todo_list_update', 66: 'timezone', 67: 'cancel_reservation', 68: 'transactions', 69: 'credit_score',
    70: 'report_fraud', 71: 'spending_history', 72: 'directions', 73: 'spelling', 74: 'insurance',
    75: 'what_is_your_name', 76: 'reminder', 77: 'where_are_you_from', 78: 'distance', 79: 'payday',
    80: 'flight_status', 81: 'find_phone', 82: 'greeting', 83: 'alarm', 84: 'order_status',
    85: 'confirm_reservation', 86: 'cook_time', 87: 'damaged_card', 88: 'reset_settings', 89: 'pin_change',
    90: 'replacement_card_duration', 91: 'new_card', 92: 'roll_dice', 93: 'income', 94: 'taxes',
    95: 'date', 96: 'who_made_you', 97: 'pto_request', 98: 'tire_pressure', 99: 'how_old_are_you',
    100: 'rollover_401k', 101: 'pto_request_status', 102: 'how_busy', 103: 'application_status', 104: 'recipe',
    105: 'calendar_update', 106: 'play_music', 107: 'yes', 108: 'direct_deposit', 109: 'credit_limit_change',
    110: 'gas', 111: 'pay_bill', 112: 'ingredients_list', 113: 'lost_luggage', 114: 'goodbye',
    115: 'what_can_i_ask_you', 116: 'book_hotel', 117: 'are_you_a_bot', 118: 'next_song', 119: 'change_speed',
    120: 'plug_type', 121: 'maybe', 122: 'w2', 123: 'oil_change_when', 124: 'thank_you', 125: 'shopping_list_update',
    126: 'pto_balance', 127: 'order_checks', 128: 'travel_alert', 129: 'fun_fact', 130: 'sync_device',
    131: 'schedule_maintenance', 132: 'apr', 133: 'transfer', 134: 'ingredient_substitution', 135: 'calories',
    136: 'current_location', 137: 'international_fees', 138: 'calculator', 139: 'definition', 140: 'next_holiday',
    141: 'update_playlist', 142: 'mpg', 143: 'min_payment', 144: 'change_user_name', 145: 'restaurant_suggestion',
    146: 'travel_notification', 147: 'cancel', 148: 'pto_used', 149: 'travel_suggestion', 150: 'change_volume'
}

In [23]:
import ipywidgets as widgets
from IPython.display import display, clear_output

def predict(user_text: str):
    """Raw text → preprocess → TF-IDF → SVD → Random Forest."""
    clean_custom = preprocess([user_text])
    custom_w = word_vec.transform(clean_custom)
    custom_c = char_vec.transform(clean_custom)
    sparse_full = sp.hstack([custom_w, custom_c])
    dense = svd.transform(sparse_full).astype(np.float32)
    return rf_model.predict(dense)[0]

text_input = widgets.Text(
    value='',
    placeholder='Type your sentence here...',
    description='Sentence:',
    layout=widgets.Layout(width='80%')
)
button = widgets.Button(
    description='Predict Intent',
    button_style='success',
    tooltip='Click to predict the intent'
)
output_area = widgets.Output()

def on_button_clicked(b):
    with output_area:
        clear_output()
        user_text = text_input.value
        if not user_text:
            print('Please enter a sentence!')
            return
        prediction_id = predict(user_text)
        intent_label = intent_mapping.get(prediction_id, 'Unknown Intent')
        print(f"You typed:      '{user_text}'")
        print(f'Predicted ID:   {prediction_id}')
        print(f'Human Intent:   [{intent_label.upper()}]')

button.on_click(on_button_clicked)
display(widgets.HBox([text_input, button]))
display(output_area)

Output()